# How to Prompt

A prompt is a contract between you and a model. The model will do exactly what the prompt says, including its omissions. This notebook covers how to write prompts that get precise, repeatable results.

## Prompt Anatomy

A complete prompt has up to four parts:

```
1. Role        Who the model is
2. Context     What it needs to know to do this task
3. Task        What it should do
4. Format      How the output should look
```

You do not always need all four. A simple task might only need the task. A complex task needs all four.

**Weak:**
```
Summarize this code.
```

**Strong:**
```
You are a senior Python engineer reviewing a pull request.

The PR adds a new auth middleware to a FastAPI app. The team uses JWT tokens.
The existing middleware is in src/middleware/auth.py.

Review the diff below and identify any security issues.

Return a numbered list. Each item: file:line, issue description, severity (low/medium/high).
```

## System Prompts

A system prompt sets the model's persistent behavior for the entire conversation. It is not a message; it is a configuration.

Use system prompts for:
- Persona and role
- Output style rules (language, tone, format)
- Hard constraints (what to never do)
- Domain context that applies to every response

Do not use system prompts for:
- Task-specific instructions (put those in the user message)
- Data that changes turn-to-turn

In [ ]:
# Example: system prompt for a code review bot

system_prompt = """
You are a senior software engineer at a fast-moving startup.
You review code for correctness, security, and simplicity.

Style rules:
- Be direct. No hedging.
- No em dashes.
- Short responses unless detail is needed.
- Use code blocks for any code you reference.

Hard constraints:
- Never suggest adding dependencies without explaining why the existing ones cannot handle it.
- Never suggest rewriting working code as part of a bug fix.
"""

print(system_prompt)

## Few-Shot Prompting

Show the model the output format by giving it examples. This is more reliable than describing the format in prose.

```
Convert these bug reports into structured tickets.

Example input: "The login button does nothing when I click it on mobile"
Example output:
Title: Login button unresponsive on mobile
Component: auth
Platform: mobile
Severity: high
Reproduction: Tap login button on mobile viewport

Now convert:
"The CSV export is missing the last row"
```

Two or three examples is usually enough. More examples reduce variance but increase token cost.

## Chain of Thought

For complex reasoning tasks, ask the model to think before answering.

```
Before answering, think through:
1. What does the code currently do?
2. What is the bug?
3. What is the minimal fix?

Then give the fix.
```

Or use extended thinking if available:

```python
response = client.messages.create(
    model="claude-opus-4-7",
    thinking={"type": "enabled", "budget_tokens": 8000},
    messages=[{"role": "user", "content": prompt}]
)
```

Chain of thought is most valuable when the answer depends on intermediate steps that are easy to get wrong.

## Format Control

Tell the model exactly what shape you want. Do not leave it to guess.

### JSON output

```
Return a JSON object with this schema:
{
  "findings": [
    {"file": string, "line": int, "issue": string, "severity": "low|medium|high"}
  ],
  "summary": string
}

Return only the JSON. No markdown, no explanation.
```

### Table output

```
Return a markdown table with columns: Service, Status, Last Deploy, Owner.
One row per service. Do not include a prose summary.
```

### Length control

```
Answer in under 100 words.
Answer in exactly 3 bullet points.
Answer with a single sentence.
```

## Prompt Templates

Reusable prompt patterns for common Mad House tasks:

In [ ]:
templates = {

    "code_review": """
Review the code at {file_path}.

Context: {context}

Check for:
1. Logic errors
2. Security issues (injection, exposed secrets, open surfaces)
3. Simplicity: is there a shorter path?

Return a numbered list. Each item: file:line, issue, severity (low/medium/high).
If there are no issues, say so in one line.
""",

    "debug": """
I have a bug. Here is the error:

{error_message}

Relevant code:

{code_snippet}

Before answering: state what the code is trying to do, then state what it is actually doing.
Then give the minimal fix.
""",

    "summarize_commits": """
Summarize these git commits into a changelog entry.

Commits:
{commits}

Group by category: Added, Changed, Fixed, Removed.
One bullet per logical change, not per commit.
Start each bullet with a past-tense verb.
No em dashes.
""",

    "explain_to_junior": """
Explain this code to a junior developer:

{code}

Cover: what problem it solves, how it works, and any non-obvious decisions.
No jargon without definition. Under 200 words.
"""

}

for name, template in templates.items():
    print(f"--- {name} ---")
    print(template[:120].strip(), "...\n")

## Anti-Patterns

| Anti-pattern | Why it fails | Fix |
|-------------|-------------|-----|
| Vague task: "make it better" | The model does not know what better means | Specify the dimension: faster, shorter, more readable, more secure |
| Over-constraining: 10 rules in the system prompt | Rules conflict or cancel each other | Keep the system prompt to 3-5 core constraints |
| Asking for too much in one prompt | Output quality degrades at scale | Break into sub-prompts, chain the outputs |
| No format instruction | The model picks a format you do not want | Always specify output shape |
| "Be creative" in a structured task | Produces inconsistent schemas | Specify the exact schema and say nothing about creativity |
| Sycophancy trap: "Does this look good?" | The model says yes | Ask "What is wrong with this?" or "What would make this fail?" |
| Burying the task | Model optimizes for what it sees first | Put the task instruction first, context second |

## The Confirm-Gate Pattern

For agent workflows where the model might take a destructive action, force it to surface its interpretation before acting:

```
Before doing anything, state:
1. What you understand the task to be (one sentence)
2. Which files you will modify
3. Which files you will not touch
4. What you will do first

Stop and wait for confirmation.
```

This costs one round-trip but prevents the model from acting on a wrong interpretation of an ambiguous task. The cost of a wrong action is always higher than the cost of one confirmation turn.